# 14 · Receive images by e-mail (IMAP)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/microsoft/SPARROW-Playground/blob/main/notebooks/14_comm_email.ipynb)

Some 4G cameras e-mail each photo. sparrow-comm polls the camera's mailbox over IMAP, keeps
only messages the camera sent (the sender must be the mailbox address itself), and stores every
image attachment. The camera name comes from a subject such as `Photo from CAM02`.

**Two ways to run this notebook**

- **Offline (default):** an in-memory stand-in for the mailbox. Nothing leaves the notebook.
- **A real mailbox:** set `SPP_IMAP_HOST`, `SPP_IMAP_ADDRESS` and `SPP_IMAP_PASSWORD` (and
  `SPP_IMAP_PORT`, default 993) before starting Jupyter. For Gmail use an
  [app password](https://support.google.com/accounts/answer/185833). The notebook places three
  demo messages in the inbox with IMAP `APPEND`, as if the camera had sent them, then polls.
  Use a test mailbox: the poller reads the whole inbox.

In [ ]:
# Setup: install sparrow-engine if it is missing (for example on Colab).
# For an NVIDIA GPU, install `sparrow-engine-gpu` instead (same Python API).
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("sparrow_engine") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sparrow-engine[video]>=0.1.31"])

import sparrow_engine as spe

print("sparrow-engine", spe.__version__, "| device:", spe.active_device())

In [ ]:
# Demo data: downloaded once into ./data and checked against a published checksum.
import hashlib
import os
import shutil
import urllib.request
import zipfile
from pathlib import Path

DATA = Path(os.environ.get("SPP_DATA_DIR", "data")).resolve()
ASSETS = {
    # Pytorch-Wildlife demo assets, CC-BY-4.0, https://zenodo.org/records/15376499
    "demo_data_det.zip": (
        "https://zenodo.org/records/15376499/files/demo_data_det.zip?download=1",
        "md5", "373e15c8e7c8894379a82086a9177763",
    ),
    "demo_data_base.zip": (
        "https://zenodo.org/records/15376499/files/demo_data_base.zip?download=1",
        "md5", "f89b9b76c753d90e4296d62d71cf1732",
    ),
    # "Yellow-Throated Vireo Song" by SwampVids, CC-BY-4.0, Wikimedia Commons
    "vireo.wav": (
        "https://upload.wikimedia.org/wikipedia/commons/4/4f/"
        "Vireo_flavifrons_-_Yellow-Throated_Vireo_Song.wav",
        "sha1", "2b472209c1b18bfc0c225975838e3a34c1e9887a",
    ),
}


def fetch(name):
    """Download an asset (once), verify it, and unzip archives into DATA."""
    url, algo, digest = ASSETS[name]
    target = DATA / name
    DATA.mkdir(parents=True, exist_ok=True)
    if not target.exists():
        print(f"downloading {name} ...")
        part = target.with_name(target.name + ".part")
        req = urllib.request.Request(url, headers={"User-Agent": "sparrow-playground"})
        with urllib.request.urlopen(req) as r, open(part, "wb") as f:
            shutil.copyfileobj(r, f)
        part.rename(target)
    h = hashlib.new(algo)
    with open(target, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    if h.hexdigest() != digest:
        target.unlink()
        raise RuntimeError(f"{name}: checksum mismatch, deleted; run the cell again")
    marker = DATA / f".{name}.extracted"
    if name.endswith(".zip") and not marker.exists():
        with zipfile.ZipFile(target) as z:
            z.extractall(DATA)
        marker.touch()
    return target

In [ ]:
# Setup: install sparrow-comm (the field-data receivers) if it is missing. It needs Python 3.12+.
# To try a build that is not on PyPI, set SPARROW_COMM_SPEC before starting Jupyter, e.g.
#   SPARROW_COMM_SPEC="/path/to/sparrow_comm-0.1.0-py3-none-any.whl[ftps]"
import os

if sys.version_info < (3, 12):
    raise RuntimeError("sparrow-comm needs Python 3.12 or newer, this kernel runs " + sys.version.split()[0])
for module, spec in [("sparrow_comm", os.environ.get("SPARROW_COMM_SPEC", "sparrow-comm[ftps]>=0.1.0")),
                     ("pyftpdlib", "pyftpdlib>=2.0"), ("OpenSSL", "pyopenssl>=24"), ("httpx", "httpx")]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", spec])

import sparrow_comm
from sparrow_comm.store import Store

print("sparrow-comm", sparrow_comm.__version__)

In [ ]:
# A fresh workspace for this notebook: a SQLite database plus a folder of received files.
# Store implements all three sparrow-comm hooks (registry, config, sink). In an app such as
# Sparrow Studio, the app's own database implements them instead.
WORK = Path("comm_workspace") / "14_email"
shutil.rmtree(WORK, ignore_errors=True)
store = Store(WORK / "comm.db", WORK / "received")
PROJECT = 1

fetch("demo_data_det.zip")
IMAGES = sorted((DATA / "data/data_example/images").rglob("*.jpg"))[:4]
print(len(IMAGES), "demo images to send:", [p.name for p in IMAGES])

## The camera's e-mails

In [ ]:
from email.message import EmailMessage
from email.utils import formatdate

ADDRESS = os.environ.get("SPP_IMAP_ADDRESS", "camera@example.org")


def camera_email(sender, subject, images):
    msg = EmailMessage()
    msg["From"], msg["To"], msg["Subject"] = sender, ADDRESS, subject
    msg["Date"] = formatdate(localtime=True)
    msg.set_content("Photo attached.")
    for img in images:
        msg.add_attachment(img.read_bytes(), maintype="image", subtype="jpeg", filename=img.name)
    return msg.as_bytes()


messages = [
    camera_email(ADDRESS, "Photo from CAM02", IMAGES[:2]),
    camera_email(ADDRESS, "Photo from CAM03", IMAGES[2:]),
    camera_email("someone@else.org", "Photo from CAM99", IMAGES[:1]),  # not the camera: ignored
]

## Connect the mailbox

In [ ]:
import imaplib

if os.environ.get("SPP_IMAP_HOST"):
    HOST, PORT = os.environ["SPP_IMAP_HOST"], int(os.environ.get("SPP_IMAP_PORT", "993"))
    PASSWORD = os.environ["SPP_IMAP_PASSWORD"]
    imap_factory = imaplib.IMAP4_SSL
    box = imaplib.IMAP4_SSL(HOST, PORT)
    box.login(ADDRESS, PASSWORD)
    for raw in messages:  # deliver the demo messages, as the camera would
        box.append("INBOX", None, None, raw)
    box.logout()
    print("real mailbox", ADDRESS, "at", HOST)
else:
    HOST, PORT, PASSWORD = "offline", 0, "offline"

    class OfflineMailbox:
        """Just enough of imaplib.IMAP4 for the poller: login, select, UID SEARCH/FETCH, logout."""

        def __init__(self, host, port):
            self.mail = {uid: raw for uid, raw in enumerate(messages, start=1)}

        def login(self, user, password):
            return "OK", [b"logged in"]

        def select(self, folder, readonly=False):
            return "OK", [str(len(self.mail)).encode()]

        def uid(self, command, *args):
            if command == "SEARCH":
                return "OK", [" ".join(map(str, self.mail)).encode()]
            uid = int(args[0])
            return "OK", [(f"{uid} (RFC822)".encode(), self.mail[uid])]

        def logout(self):
            return "BYE", []

    imap_factory = OfflineMailbox
    print("offline mailbox with", len(messages), "messages")

## Poll

The operator stores the mailbox password encrypted, so `Store` needs a secret key
(`sparrow-comm secret-key` prints one). The poller remembers the last IMAP UID it read, so a
second poll fetches nothing new.

In [ ]:
from cryptography.fernet import Fernet
from sparrow_comm.channels import imap

store = Store(WORK / "comm.db", WORK / "received", secret_key=Fernet.generate_key().decode())
store.set_imap_account(PROJECT, ADDRESS, PASSWORD)

print("first poll:", imap.poll_once(store, store, host=HOST, port=PORT, imap_factory=imap_factory), "images")
print("second poll:", imap.poll_once(store, store, host=HOST, port=PORT, imap_factory=imap_factory), "images")

In [ ]:
import pandas as pd

received = [r for r in store.items(PROJECT) if r["kind"] == "image"]
pd.DataFrame(received)[["id", "channel", "camera_name", "original_name", "captured_at", "path"]]

## Process with sparrow-engine

In [ ]:
# Models are installed into SPARROW_ENGINE_MODEL_DIR (default ~/.sparrow-engine/models).
# Already-installed models are skipped. Inference never downloads anything by itself.
for model_id in ["MDV6-yolov10-c"]:
    spe.download_model(model_id)

In [ ]:
# Run MegaDetector V6 on the files exactly where sparrow-comm stored them.
paths = [store.data_dir / r["path"] for r in received]
results = spe.detect(paths, model="MDV6-yolov10-c", threshold=0.3)
by_path = {Path(r.path).resolve(): r for r in results}

rows = []
for r, p in zip(received, paths):
    dets = by_path[p.resolve()].detections
    rows.append({
        "channel": r["channel"],
        "camera": r["camera_name"],
        "file": r["original_name"],
        "captured_at": r["captured_at"],
        "detections": len(dets),
        "labels": sorted({d.label for d in dets}),
        "max_conf": round(max((d.confidence for d in dets), default=0.0), 2),
    })
pd.DataFrame(rows)

In [ ]:
from IPython.display import Image, display

for png in spe.visualize(results[:2]):
    display(Image(data=png, width=480))

## Run it for real

```bash
export SPARROW_COMM_SECRET_KEY=$(sparrow-comm secret-key)
sparrow-comm --db comm.db --data-dir received imap set 1 camera@example.org   # prompts for the password
sparrow-comm --db comm.db --data-dir received poll imap --host imap.gmail.com
```